In [ ]:
import torch
import matplotlib.pyplot as plt

from awsa import IMAGE_SIZE
from awsa.data import load_images, load_metadata
from awsa.models import load_resnet50, imagenet_categories
from awsa.saliency import gradcam

In [ ]:
meta = load_metadata().iloc[0]

x = load_images([meta.ImageId])  # preprocessed 224x224 (scripts/prepare_data.py)
img = x[0]
y = torch.tensor([int(meta.label)])

model = load_resnet50("cpu")
categories = imagenet_categories()

with torch.no_grad():
    pred = model(x).argmax(1)

cam = gradcam(model, x, y)

print("True class:", categories[y.item()])
print("Predicted class:", categories[pred.item()])
print("Grad-CAM shape:", tuple(cam.shape))

In [ ]:
import torch.nn.functional as F

cam_up = F.interpolate(
    cam,
    size=(IMAGE_SIZE, IMAGE_SIZE),
    mode="bilinear",
    align_corners=False,
)[0, 0]

img_np = img.permute(1, 2, 0).numpy()

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(img_np)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(cam_up.numpy(), cmap="jet")
plt.title("Grad-CAM")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img_np)
plt.imshow(cam_up.numpy(), cmap="jet", alpha=0.45)
plt.title("Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from awsa.masks import top_fraction_mask, blocks_to_pixels

area = 0.25

high_mask = top_fraction_mask(cam, area)
high_px = blocks_to_pixels(high_mask)

print("Block mask shape:", tuple(high_mask.shape))
print("Selected blocks:", high_mask.sum().item())
print("Expected blocks:", round(area * 28 * 28))
print("Pixel mask shape:", tuple(high_px.shape))

In [ ]:
mask_np = high_px[0, 0].numpy()

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.imshow(mask_np, cmap="Reds", alpha=0.35)
plt.title("Top 25% Saliency Blocks")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.contour(mask_np, levels=[0.5], colors="red", linewidths=1)
plt.title("Selected Block Boundaries")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import importlib
import awsa.masks as masks

importlib.reload(masks)

low_mask = masks.bottom_fraction_mask(
    cam,
    area=0.25,
    exclude=high_mask,
)

low_px = masks.blocks_to_pixels(low_mask)

print("High-saliency blocks:", high_mask.sum().item())
print("Low-saliency blocks:", low_mask.sum().item())
print("Overlap:", (high_mask & low_mask).sum().item())

In [ ]:
high_np = masks.blocks_to_pixels(high_mask)[0, 0].numpy()
low_np = low_px[0, 0].numpy()

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(img_np)
plt.contour(high_np, levels=[0.5], colors="red", linewidths=1.2)
plt.title("High-Saliency Region")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(img_np)
plt.contour(low_np, levels=[0.5], colors="blue", linewidths=1.2)
plt.title("Low-Saliency Region")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img_np)
plt.contour(high_np, levels=[0.5], colors="red", linewidths=1.2)
plt.contour(low_np, levels=[0.5], colors="blue", linewidths=1.2)
plt.title("Separated Allocation")
plt.axis("off")

plt.tight_layout()
plt.show()